<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/4_01_siliver_utilities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 4.01 Silver Utilities (Data Transforms)
We return back to the bank - this time to make a DAG. As a reminder, our system receives card transactions from a processor, weekly customer snapshots from its CRM, and daily exchange rates. Management want spend in pounds by month and customer segment. The raw (bronze) data is messy. This notebook gives you the 20 steps that clean and join it into the silver layer. **NOTE: 20 steps is not exhaustive and these transforms could be done differently based on how we best interpret the business case.**

*Your task (on paper, in groups).* Draw a DAG with one box per step. Draw an arrow from step A to step B if B cannot run until A has finished. Use each step's *Needs* and *Adds* to decide. **NOTE: our previous convention of a function including a description, INPUT and OUTPUT is probably the better one to follow. We have replaced with *Needs* and *Adds* simply to make the DAG task simpler**

Things to settle as a group: which steps are independent of each other and can be parallelised, which must wait, and where the three sources first meet.

Nothing here needs coding. Code to import the data is here purely if you want to experiment later. Most of the silver transforms show `pass` meaning they have no yet been implemented (like if you don't know the answer in a quiz you say "pass"). An asychronous task/practice test next week will let you put this all together.

In [ ]:
# ---- Setup: run this cell first ----
import glob, os, urllib.request, zipfile, shutil
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dagsession/card_spend_dbt.zip"
DATA_DIR = "/content/card-spend"

# Force clean extraction every time to ensure integrity of files
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

zip_path = "/content/card_spend_dbt.zip"
urllib.request.urlretrieve(DATA_URL, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

def find(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"Missing {name}. Checked {DATA_DIR}. Ask for help.")
    return hits[0]

bronze_tx = pd.read_csv(find("bronze_card_transactions.csv"), dtype={"ref_txn_id": "string"})
bronze_cu = pd.read_csv(find("bronze_customer_master.csv"))
bronze_fx = pd.read_csv(find("bronze_fx_rates.csv"))
print(f"card_transactions: {len(bronze_tx):,} rows | customer_master: {len(bronze_cu):,} rows | fx_rates: {len(bronze_fx):,} rows")

This cell writes the file `utilities.py` (in the same fashion as we saw in 3_01). You do not need to read the code; the descriptions are what matter.

In [ ]:
%%writefile utilities.py
"""
utilities.py: silver-layer transform steps for the bank card-spend case.

The case: a retail bank with three source systems.
  card_transactions  card processor files (one row per transaction event per file)
  customer_master    weekly CRM snapshots (one row per card per snapshot)
  fx_rates           daily exchange rates to GBP

How to read this file
  * Each function is one step (a building block) that works on a pandas DataFrame.
  * "Needs" lists what a step reads. "Adds" lists what it creates or changes.
    Use these to work out which steps must run before which, and which are independent.
  * The task is to draw the order of the steps, not to run them.

Conventions: steps take a DataFrame and return a new DataFrame (the input is not changed).

Index of the 20 steps
  A. Transactions
     drop_exact_duplicates
     standardise_currency_codes
     standardise_merchant_text
     parse_timestamps
     derive_transaction_date_and_month
     keep_latest_status_per_transaction
     flag_reversals
     flag_zero_amount
     link_reversals_to_original
     flag_foreign_currency
  B. Customers
     standardise_segment_labels
     latest_customer_snapshot_per_card
  C. Exchange rates
     build_fx_calendar
     fill_missing_fx_rates
     add_gbp_rate
     flag_filled_fx_rates
  D. Joining and enriching
     add_segment_from_latest_snapshot
     add_fx_rate_to_transactions
     add_gbp_amount
  E. Check
     check_one_row_per_transaction
"""

import pandas as pd

STATUS_RANK = {"PENDING": 0, "POSTED": 1, "REVERSED": 2}


# =====================================================================
# A. Transactions
# =====================================================================


def drop_exact_duplicates(df):
    """[built] Remove rows that are identical in every column.

    Why: the processor sometimes sends a whole daily file twice.
    Needs: the raw transactions, as delivered (before any columns are added)
    Adds:  nothing (fewer rows)
    """
    return df.drop_duplicates().reset_index(drop=True)


def standardise_currency_codes(df, col="currency"):
    """[built] Trim spaces and upper-case currency codes ('gbp ' becomes 'GBP').

    Needs: currency
    Adds:  currency (cleaned, same column)
    """
    out = df.copy()
    out[col] = out[col].str.strip().str.upper()
    return out


def standardise_merchant_text(df):
    """[built] Tidy merchant names: trim, collapse repeated spaces, upper-case.
    The original text is kept in `merchant`.

    Needs: merchant
    Adds:  merchant_clean
    """
    pass


def parse_timestamps(df):
    """[built] Turn the text timestamp into a real datetime.
    Two formats arrive mixed: 'YYYY-MM-DD hh:mm:ss' and day-first 'dd/mm/yyyy hh:mm'.
    Anything with a '/' is read day first (an assumption worth stating).

    Needs: txn_timestamp
    Adds:  ts
    """
    out = df.copy()
    s = out["txn_timestamp"]
    day_first = s.str.contains("/", regex=False)
    ts = pd.Series(pd.NaT, index=out.index, dtype="datetime64[ns]")
    ts[day_first] = pd.to_datetime(s[day_first], format="%d/%m/%Y %H:%M")
    ts[~day_first] = pd.to_datetime(s[~day_first], format="%Y-%m-%d %H:%M:%S")
    out["ts"] = ts
    return out


def derive_transaction_date_and_month(df):
    """[built] Work out the transaction date and its month (YYYY-MM).
    Decision recorded here: the month is the month the purchase was MADE, not the month
    the file arrived.

    Needs: ts
    Adds:  tdate, month
    """
    pass


def keep_latest_status_per_transaction(df):
    """[built] Keep one row per transaction id: the latest known status.
    Latest = the most recent file date; if two rows share a file date, the higher
    status wins (PENDING < POSTED < REVERSED).

    Needs: txn_id, file_date, status
    Adds:  nothing (one row per txn_id)
    """
    out = df.copy()
    out["_rank"] = out["status"].map(STATUS_RANK)
    out = out.sort_values(["txn_id", "file_date", "_rank"])
    out = out.groupby("txn_id").tail(1).drop(columns="_rank")
    return out.reset_index(drop=True)


def flag_reversals(df):
    """[built] Mark rows that are reversals (a refund recorded as its own row).

    Needs: status
    Adds:  is_reversal (True/False)
    """
    pass


def flag_zero_amount(df):
    """[built] Mark transactions with an amount of exactly zero (keep them, flagged).

    Needs: amount
    Adds:  zero_amount (True/False)
    """
    pass


def link_reversals_to_original(df):
    """[not built] Connect each reversal row to the original purchase it cancels,
    and flag reversals whose original cannot be found.

    Needs: txn_id, ref_txn_id, is_reversal (one row per txn_id, so originals can be found)
    Adds:  original_found (True/False), original_amount
    """
    pass


def flag_foreign_currency(df):
    """[not built] Mark transactions made in a currency other than GBP.

    Needs: currency
    Adds:  is_foreign_currency (True/False)
    """
    pass


# =====================================================================
# B. Customers
# =====================================================================


def standardise_segment_labels(df):
    """[built] Make segment labels consistent ('premium', 'PREMIUM ' become 'Premium').

    Needs: segment
    Adds:  segment (cleaned, same column)
    """
    pass


def latest_customer_snapshot_per_card(df):
    """[built] Reduce the weekly snapshots to one row per card: the most recent record.
    Decision recorded here: 'segment' means the segment NOW (latest snapshot).

    Needs: card_id, snapshot_date, customer_id, segment, account_status
    Adds:  nothing (one row per card)
    """
    pass


# =====================================================================
# C. Exchange rates
# =====================================================================


def build_fx_calendar(fx, end_date):
    """[built] Lay the rates out on a full calendar: one row per day from the first
    published rate to `end_date`, one column per currency. Days with no published rate
    (weekends, holidays) are left empty at this stage.

    Needs: rate_date, currency, rate_to_gbp
    Adds:  a table indexed by day, one column per currency
    """
    pass


def fill_missing_fx_rates(fx_wide):
    """[built] Fill days with no published rate with the last published rate.
    Decision recorded here: weekends and holidays use the previous published day.

    Needs: the calendar table from build_fx_calendar
    Adds:  nothing (empty days are filled)
    """
    pass


def add_gbp_rate(fx_wide):
    """[built] Add GBP itself with a rate of 1.0, so GBP transactions convert like any other.

    Needs: the calendar table
    Adds:  a GBP column (all 1.0)
    """
    pass


def flag_filled_fx_rates(fx_wide):
    """[not built] Record which days used a carried-forward rate instead of a published
    one, so a user can tell a real rate from a filled one.

    Needs: the calendar table BEFORE it is filled
    Adds:  rate_is_filled (True/False, per day)
    """
    pass


# =====================================================================
# D. Joining and enriching
# =====================================================================


def add_segment_from_latest_snapshot(tx, cards):
    """[built] Attach each transaction's customer segment (latest snapshot).
    Cards missing from the customer master get segment 'Unknown' and are flagged,
    not dropped, so totals still reconcile.

    Needs: tx: card_id | cards: card_id, segment (one row per card)
    Adds:  segment, customer_known (True/False)
    """
    pass


def add_fx_rate_to_transactions(tx, fx_wide):
    """[built] Look up the exchange rate for each transaction: the rate on the
    transaction DATE, for the transaction's currency.

    Needs: tx: tdate, currency | fx_wide: the filled calendar table including GBP
    Adds:  rate
    """
    pass


def add_gbp_amount(df):
    """[built] Convert each amount to pounds.

    Needs: amount, rate
    Adds:  amount_gbp
    """
    pass


# =====================================================================
# E. Check
# =====================================================================


def check_one_row_per_transaction(df):
    """[not built] Check that txn_id is unique and never empty in the finished silver table.

    Needs: txn_id, in the finished table
    Adds:  nothing (passes silently or reports the problem)
    """
    pass
